# 📊 Swiss Legal Retrieval — Data Exploration

Pure data investigation for the [LLM Agentic Legal Retrieval competition](https://www.kaggle.com/competitions/llm-agentic-legal-information-retrieval).
No retrieval code here — just looking at the data carefully before building
anything, and documenting what I found so future decisions are justified.

Companion notebooks:
- 🗃️ [Corpus utility](link) — builds `corpus.parquet`
- 🔍 [Weaviate utility](link) — pre-built vector index
- 🎯 [Retrieval + submission](link) — the actual pipeline

## 👋 About me

Publishing this as a community hobby project — I'm **not eligible for the
prize pool**, Feel free to fork, modify, learn from it.

**I'm an Agentic AI Developer.** I work across the whole modern LLM
application stack — agents, retrieval, fine-tuning, ... .

**I'm looking for work** 🔍. If you're hiring for Agentic AI / LLM
application engineering roles, please reach out:

📧 **re.azami@gmail.com**

Remote, hybrid, or on-site — all welcome. Happy to collaborate on projects too.

## 🗺️ What I investigated

- 👀 **Query structure** — shape of train.csv, what queries look like
- 🔢 **Citation count per query** — drove the "dynamic k" decision
- ⚖️ **Train vs val citation types** — caught a 98.8% → 59% distribution shift
- 🎯 **Format match between gold citations and corpus** — found the 29% gap
- 🔬 **Where did the missing 29% go?** — dug into raw source files
- 🔗 **Article-level vs paragraph-level granularity** — confirmed with the competition host
- 📐 **F1 ceiling analysis** — 71% → 98% with smart matching, 2% truly unreachable

In [1]:
import numpy as np
import pyarrow.parquet as pq
from tqdm.auto import tqdm
import sys, subprocess
import hashlib, shutil, os, stat
from pathlib import Path
import time
import pandas as pd


## 📊 Investigating the data

Before writing any retrieval code, I needed to understand what I'd be scored
on. The cells below answer three questions: what do queries look like, how
many citations does each one expect, and what kind of sources get cited.
Each answer shaped a decision in the retrieval pipeline.

### 👀 What does train.csv look like?

Column names, row counts, a few examples. You can't build a pipeline against
data you haven't looked at.

In [2]:
# Inspect train.csv — understand what retrieval needs to produce

TRAIN_PATH = "/kaggle/input/competitions/llm-agentic-legal-information-retrieval/train.csv"
train = pd.read_csv(TRAIN_PATH)

print(f"Shape: {train.shape}")
print(f"Columns: {list(train.columns)}")
print(f"\nDtypes:")
print(train.dtypes)
print(f"\nFirst 3 rows:")
pd.set_option("display.max_colwidth", None)
for i, row in train.head(3).iterrows():
    print(f"\n--- Row {i} ---")
    for col in train.columns:
        val = str(row[col])
        if len(val) > 300:
            val = val[:300] + f"... [{len(str(row[col]))} chars total]"
        print(f"  {col}: {val}")

Shape: (1139, 3)
Columns: ['query_id', 'query', 'gold_citations']

Dtypes:
query_id          object
query             object
gold_citations    object
dtype: object

First 3 rows:

--- Row 0 ---
  query_id: train_0001
  query: Die A AG betreibt seit den 1970er-Jahren auf der Parzelle Nr. yyy (Wohn- und Gewerbezone) ein Recyclingunternehmen. Das Unternehmen verarbeitet vorwiegend Metallschrott. Dieser wird im Aussenbereich des Betriebsgeländes zwischengelagert und dort von einem fest instal lierten Metallschredder für die ... [1495 chars total]
  gold_citations: Art. 10a Abs. 1 USG;Art. 2 Abs. 1 UVPV;Art. 10a Abs. 1 UVG

--- Row 1 ---
  query_id: train_0002
  query: Die Alpha Consulting AG kann nun ihr Grundstück neu direkt an die soeben erstellte Kana lisationsleitung in der angrenzenden neuen Quartierstrasse anschliessen, so dass sie die von der Grunddienstbarkeit erfasste Abwasserleitung (vgl. Sachverhalt Teil 1) ausser Betrieb nimmt und ihr Grundstück jetzt... [1376 chars total]
  go

In [3]:
# Is each row one (query, citation) pair, or is gold_citation a list of citations?
print("Unique query_ids:", train["query_id"].nunique())
print("Total rows:", len(train))
print(f"Rows per query: {len(train) / train['query_id'].nunique():.2f}")
print()

# Look at a few gold_citation values verbatim
print("First 5 gold_citation values, raw:")
for i, val in enumerate(train["gold_citations"].head(5)):
    print(f"  [{i}] type={type(val).__name__}  len={len(str(val))}")
    print(f"       {val!r}")
    print()

Unique query_ids: 1139
Total rows: 1139
Rows per query: 1.00

First 5 gold_citation values, raw:
  [0] type=str  len=58
       'Art. 10a Abs. 1 USG;Art. 2 Abs. 1 UVPV;Art. 10a Abs. 1 UVG'

  [1] type=str  len=171
       'Art. 975 ZGB;Art. 974 Abs. 2 ZGB;Art. 973 ZGB;Art. 661 ZGB;Art. 956a ZGB;Art. 956a Abs. 3 ZGB;Art. 955 Abs. 1 ZGB;Art. 976 ZGB;Art. 976a ZGB;Art. 60 OR;Art. 955 Abs. 2 ZGB'

  [2] type=str  len=14
       'Art. 264m StGB'

  [3] type=str  len=83
       'Art. 176 Abs. 1 IPRG;Art. 186 Abs. 1 IPRG;Art. 186 Abs. 3 IPRG;Art. 190 Abs. 3 IPRG'

  [4] type=str  len=99
       'Art. 93 Abs. 1 BGG;Art. 89 Abs. 1 BGG;Art. 89 Abs. 2 BGG;Art. 50 BV;Art. 95 BGG;Art. 106 Abs. 2 BGG'



🔑 **Finding**: 1,139 training queries, one row per query, with
`gold_citations` packed as a single semicolon-separated string. Each query
is a multi-paragraph German legal exam scenario (~1,400 chars on average).
Queries are in German. The test set might be in English (the competition description suggests a
cross-lingual setup).

### 🔢 How many citations per query?

The scoring metric is macro F1 across queries. If I predict too few citations
per query I lose recall; too many and I tank precision. So: how many gold
citations does each training query actually have?

In [4]:
# How many gold citations per query? This determines optimal k.


counts = train["gold_citations"].astype(str).apply(lambda s: len([c for c in s.split(";") if c.strip()]))
print(f"Counts: \n{counts}")
print(f"Gold citations per query:")
print(f"  mean:   {counts.mean():.2f}")
print(f"  median: {counts.median():.0f}")
print(f"  std:    {counts.std():.2f}")
print(f"\n  percentiles:")
for p in [10, 25, 50, 75, 90, 95, 99, 100]:
    print(f"    p{p:>3}: {int(np.percentile(counts, p))}")
print(f"\n  distribution:")
print(counts.value_counts().sort_index().head(44))

Counts: 
0        3
1       11
2        1
3        4
4        6
        ..
1134     5
1135     1
1136     9
1137     1
1138     6
Name: gold_citations, Length: 1139, dtype: int64
Gold citations per query:
  mean:   4.09
  median: 2
  std:    4.51

  percentiles:
    p 10: 1
    p 25: 1
    p 50: 2
    p 75: 5
    p 90: 9
    p 95: 12
    p 99: 21
    p100: 44

  distribution:
gold_citations
1     341
2     231
3     149
4      85
5      80
6      50
7      31
8      31
9      29
10     28
11     14
12     13
13     11
14      8
15      9
16      3
17      4
18      2
19      5
20      2
21      2
22      1
23      1
24      2
27      2
29      1
38      3
44      1
Name: count, dtype: int64


🔑 **Finding**: mean is 4.09 per query but the distribution is extremely
spread — 30% of queries have exactly 1 citation, 20% have 2, but p95 is 12,
p99 is 21, and the max is 44. A fixed `k` would catastrophically under- or
overshoot on most queries.

**Decision**: use a **score threshold** (in v1) instead of fixed k. Retrieve top-50
candidates, keep all whose hybrid score is above a threshold tuned on
train.csv. Simple, no training, directly optimizes the F1 I'm being scored on.

### ⚖️ Laws or court decisions?

The competition provides two corpora: Swiss federal laws (`laws_de.csv`,
~176k rows) and court decision considerations (`court_considerations.csv`,
~2.5M rows). Before indexing both at significant cost, I want to know which
kind of source actually gets cited in training answers.

In [5]:
# Are gold citations mostly laws (Art. ...) or mostly decisions (BGE ...)?
all_golds = []
for val in train["gold_citations"].astype(str):
    all_golds.extend(c.strip() for c in val.split(";") if c.strip())

n_total = len(all_golds)
n_laws = sum(1 for c in all_golds if c.startswith("Art."))
n_bge  = sum(1 for c in all_golds if c.startswith("BGE"))
n_other = n_total - n_laws - n_bge

print(f"Total gold citations in train: {n_total:,}")
print(f"  Laws (Art. ...):        {n_laws:>6,} ({100*n_laws/n_total:.1f}%)")
print(f"  BGE decisions:          {n_bge:>6,} ({100*n_bge/n_total:.1f}%)")
print(f"  Other:                  {n_other:>6,} ({100*n_other/n_total:.1f}%)")

print("\n10 random gold citations:")
import random
random.seed(42)
for c in random.sample(all_golds, 10):
    print(f"  {c!r}")

Total gold citations in train: 4,659
  Laws (Art. ...):         4,602 (98.8%)
  BGE decisions:              57 (1.2%)
  Other:                       0 (0.0%)

10 random gold citations:
  'Art. 963 Abs. 1 ZGB'
  'Art. 5 Abs. 1 FIDLEG'
  'Art. 7 Abs. 2 PatG'
  'Art. 559 ZGB'
  'Art. 3 Abs. 1 KVG'
  'Art. 83 Abs. 2 GBV'
  'Art. 69 ZGB'
  'Art. 9 SebG'
  'Art. 14 Abs. 1 StromVG'
  'Art. 27 DSG'


🔑 **Finding**: **98.8% laws, 1.2% BGE decisions.** Essentially a statute-only
task.

**Decision**: filter the corpus to laws only at ingest time. This cuts ingest
from ~2.9M chunks to ~173k chunks — roughly 17× less work per submission run,
cleaner retrieval (no decision noise crowding out relevant laws), and a
theoretical F1 ceiling of 98.8% in exchange for the 1.2% I can no longer
reach. The parquet still contains decisions for future. My v1 submission notebook just filters them out on load.

### 🧭 Can I cheat on the 1.2%?

Before fully committing to "ignore decisions," I want to check whether the
BGE citations in training are a concentrated pattern I could cover cheaply —
like if the same 5 landmark decisions appeared many times each. If they
were, it'd be worth a small lookup table to grab that 1.2% of F1 for free.

In [6]:
# How unique are the BGE decision citations in train?
bge_citations = [c for c in all_golds if c.startswith("BGE")]

print(f"Total BGE mentions: {len(bge_citations)}")
print(f"Unique BGE strings: {len(set(bge_citations))}")
print()

# Show them all with counts
from collections import Counter
counter = Counter(bge_citations)
print("All unique BGE citations (most common first):")
for cit, count in counter.most_common():
    print(f"  [{count}×] {cit!r}")

Total BGE mentions: 57
Unique BGE strings: 52

All unique BGE citations (most common first):
  [3×] 'BGE 121 IV 94 E. 1b'
  [3×] 'BGE 127 III 543 E. 2c'
  [2×] 'BGE 137 I 340 E. 3.1'
  [1×] 'BGE 141 IV 20 E. 1.1.4'
  [1×] 'BGE 141 IV 289 E. 2.3'
  [1×] 'BGE 134 III 565 E. 3.2'
  [1×] 'BGE 129 II 100 E. 3.5'
  [1×] 'BGE 137 III 481 E. 2.1'
  [1×] 'BGE 136 III 322 E. 3'
  [1×] 'BGE 132 III 564 E. 6.2'
  [1×] 'BGE 132 III 321 E. 2.2.1'
  [1×] 'BGE 129 III 331 E. 2.1'
  [1×] 'BGE 132 III 564 E. 3.1.2'
  [1×] 'BGE 132 III 564 E. 3.1.3'
  [1×] 'BGE 132 III 342 E. 2.3.3'
  [1×] 'BGE 132 III 564 E. 3.2.2'
  [1×] 'BGE 128 III 180 E. 2c'
  [1×] 'BGE 132 III 564 E. 3.2.3'
  [1×] 'BGE 141 III 112 E. 5.2.3'
  [1×] 'BGE 131 III 306 E. 3.1.1'
  [1×] 'BGE 132 III 564 E. 3.1'
  [1×] 'BGE 131 III 306 E. 3.1.2'
  [1×] 'BGE 132 III 564 E. 3.2.1'
  [1×] 'BGE 141 III 112 E. 5.2.1'
  [1×] 'BGE 132 III 342 E. 2.3.1'
  [1×] 'BGE 132 III 342 E. 4.1'
  [1×] 'BGE 143 IV 9 E. 2.3.1'
  [1×] 'BGE 129 III 715 E. 4.2'

🔑 **Finding**: 57 BGE mentions in training, **52 unique citations**. The
most-cited decision appears 3 times. These are pinpoint citations at the
sub-consideration level (e.g. `BGE 132 III 564 E. 3.1.2`) — a scattered long
tail with no reusable pattern.

**Decision**: confirmed — ignore decisions. There's no cheap win available,
and trying to retrieve 52 specific sub-paragraphs from a 2.5M-row corpus would
add more noise than the 1.2% upside is worth. Move on.

### 🔁 Wait — does val.csv look the same?

Earlier I found train.csv was 98.8% laws and decided to filter the corpus to
laws only. Before committing to that, I should check val.csv too. If train
and val have different citation distributions, "laws only" might be a
disastrous decision for the actual scored set.

In [7]:

# Load val.csv
val_df = pd.read_csv('/kaggle/input/competitions/llm-agentic-legal-information-retrieval/val.csv')

# Flatten the semicolon-separated citations
val_citations = val_df['gold_citations'].dropna().str.split(';').explode().str.strip()

total_val = len(val_citations)
val_laws = val_citations[val_citations.str.startswith('Art.', na=False)]
val_decisions = val_citations[val_citations.str.startswith('BGE', na=False)]
val_other = val_citations[~val_citations.str.startswith('Art.', na=False) & ~val_citations.str.startswith('BGE', na=False)]

law_pct = (len(val_laws) / total_val) * 100
dec_pct = (len(val_decisions) / total_val) * 100
other_pct = (len(val_other) / total_val) * 100

print(f"Total citations in val.csv: {total_val}")
print(f"Laws ('Art.'): {len(val_laws)} ({law_pct:.2f}%)")
print(f"Decisions ('BGE'): {len(val_decisions)} ({dec_pct:.2f}%)")
print(f"Other/Unknown: {len(val_other)} ({other_pct:.2f}%)")

if len(val_decisions) > 0:
    print(f"\nUnique Decisions in val.csv: {val_decisions.nunique()}")
    print("Examples:")
    print(val_decisions.unique()[:5])

if len(val_other) > 0:
    print(f"\n⚠️ WARNING: Found unexpected citation formats in 'Other':")
    print(val_other.unique()[:5])

Total citations in val.csv: 251
Laws ('Art.'): 149 (59.36%)
Decisions ('BGE'): 69 (27.49%)
Other/Unknown: 33 (13.15%)

Unique Decisions in val.csv: 68
Examples:
['BGE 137 IV 122 E. 6.2' 'BGE 137 IV 122 E. 6.4' 'BGE 137 IV 122 E. 4.2'
 'BGE 132 I 21 E. 3.2' 'BGE 132 I 21 E. 3.2.2']

⚠️ WARNING: Found unexpected citation formats in 'Other':
['1B_210/2023 E. 4.1' '1B_536/2018 E. 5.1' '1B_90/2021 E. 2.1'
 '1B_90/2021 E. 2.4' '7B_496/2025 E. 3.2']


🔑 **Finding**: Completely different distribution — **59% laws, 27% BGE
decisions, 13% court file numbers** (a third format I hadn't seen at all in
train, like `1B_210/2023 E. 4.1`).

**Decision reversed**: filtering to laws-only would have hard-capped my
val/test F1 at ~59%. Drop the filter — **ingest the full corpus**, all
2.9M chunks. The cost is ~30 extra minutes per submission run. The benefit
is access to 41% of citations I'd otherwise miss completely.

**Lesson learned**: train and val can have meaningfully different
distributions in this competition. Decisions made from train alone are
suspect until verified against val.

## 🏗️ Setup

Attach the utility notebook (which contains `corpus.parquet` + bge-m3 model
+ Weaviate binary + offline wheels) and install packages from the staged
wheels with no network calls. The submission notebook from here on runs
fully offline.

In [8]:
WHEELS_UTIL = Path("/kaggle/usr/lib/notebooks/rezaazami/utility_swiss_legal_wheels_and_models")
CORPUS_UTIL = Path("/kaggle/usr/lib/notebooks/rezaazami/utility_swiss_legal_corpus")

assert WHEELS_UTIL.exists(), f"Wheels notebook not attached at {WHEELS_UTIL}"
assert CORPUS_UTIL.exists(), f"Corpus notebook not attached at {CORPUS_UTIL}"

PARQUET    = CORPUS_UTIL / "corpus.parquet"
WHEELS     = WHEELS_UTIL / "wheels"
MODEL_PATH = WHEELS_UTIL / "models" / "bge-m3"

for p in [PARQUET, WHEELS, MODEL_PATH]:
    assert p.exists(), f"Missing: {p}"

print("Utility artifacts:")
print(f"  corpus.parquet  {os.path.getsize(PARQUET)/1e9:.2f} GB")
print(f"  wheels          {WHEELS}")
print(f"  bge-m3 model    {MODEL_PATH}")

# Install offline from the staged wheels
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "--no-index", "--find-links", str(WHEELS),
    "sentence-transformers==5.4.0",
])
print("\nPackages installed offline from utility wheels")

Utility artifacts:
  corpus.parquet  6.45 GB
  wheels          /kaggle/usr/lib/notebooks/rezaazami/utility_swiss_legal_wheels_and_models/wheels
  bge-m3 model    /kaggle/usr/lib/notebooks/rezaazami/utility_swiss_legal_wheels_and_models/models/bge-m3

Packages installed offline from utility wheels


🔑 **Confirmed**: all four artifacts present, packages installed offline.
6.42 GB parquet ready to go.

### 🎯 Do the gold citations actually exist in the corpus?

The #1 silent failure mode in retrieval competitions: gold citation strings
might use a different format than corpus citation strings, making correct
predictions look wrong to the scorer. Before tuning anything, verify both
train and val gold citations exist *verbatim* in `corpus.parquet`.

In [9]:

print("Loading train and val CSVs...")
train_df = pd.read_csv('/kaggle/input/competitions/llm-agentic-legal-information-retrieval/train.csv')
val_df = pd.read_csv('/kaggle/input/competitions/llm-agentic-legal-information-retrieval/val.csv')

print("Loading corpus citations...")
parquet_file = pq.ParquetFile(PARQUET)
corpus_citations = []

# Read in batches to show a progress bar for the slow step
for batch in tqdm(parquet_file.iter_batches(columns=['citation']), total=parquet_file.num_row_groups, desc="Reading Parquet"):
    corpus_citations.extend(batch.to_pandas()['citation'].tolist())
    
corpus_set = set(corpus_citations)

print("Flattening gold citations...")
train_golds = train_df['gold_citations'].dropna().str.split(';').explode().str.strip()
val_golds = val_df['gold_citations'].dropna().str.split(';').explode().str.strip()

unique_train_golds = train_golds.unique()
unique_val_golds = val_golds.unique()

print("Calculating intersections...")
# These vectorised operations take <1 second
train_matches = pd.Series(unique_train_golds).isin(corpus_set)
val_matches = pd.Series(unique_val_golds).isin(corpus_set)

train_match_rate = train_matches.mean() * 100
val_match_rate = val_matches.mean() * 100

print(f"\n--- Train Data Match Check ---")
print(f"Total unique gold citations in train: {len(unique_train_golds)}")
print(f"Matched in corpus: {train_matches.sum()}")
print(f"Match rate: {train_match_rate:.2f}%")

if train_match_rate < 100:
    missing_train = pd.Series(unique_train_golds)[~train_matches]
    print(f"❌ Missing examples (Train): {missing_train.head(5).tolist()}")

print(f"\n--- Validation Data Match Check ---")
print(f"Total unique gold citations in val: {len(unique_val_golds)}")
print(f"Matched in corpus: {val_matches.sum()}")
print(f"Match rate: {val_match_rate:.2f}%")

if val_match_rate < 100:
    missing_val = pd.Series(unique_val_golds)[~val_matches]
    print(f"❌ Missing examples (Val): {missing_val.head(5).tolist()}")

Loading train and val CSVs...
Loading corpus citations...


Reading Parquet:   0%|          | 0/30 [00:00<?, ?it/s]

Flattening gold citations...
Calculating intersections...

--- Train Data Match Check ---
Total unique gold citations in train: 2695
Matched in corpus: 1926
Match rate: 71.47%
❌ Missing examples (Train): ['Art. 10a Abs. 1 UVG', 'Art. 975 ZGB', 'Art. 973 ZGB', 'Art. 956a ZGB', 'Art. 976a ZGB']

--- Validation Data Match Check ---
Total unique gold citations in val: 222
Matched in corpus: 222
Match rate: 100.00%


🔑 **Finding**: Train matches **only 71.24%** (1920 of 2695 unique gold
citations). Val matches **100%** (222 of 222).

The asymmetry is suspicious. If this were a uniform format mismatch, both
would match poorly. The fact that val is perfect suggests the missing 29% of
train aren't a format problem with our pipeline — they're something specific
about train. Time to dig deeper.

### 🔬 Where did the missing 29% go?

Two hypotheses for why those train citations aren't in our parquet:

- **A**: My utility notebook's filtering (min char length, "Aufgehoben"
  filter) accidentally dropped them during the build.
- **B**: They never existed in the raw `laws_de.csv` to begin with.

Easy test: check if the missing train citations exist in the raw file
*before* any of my processing touched it.

In [10]:

print("Loading raw laws_de.csv (citation column only)...")
raw_laws = pd.read_csv('/kaggle/input/competitions/llm-agentic-legal-information-retrieval/laws_de.csv', usecols=['citation'])
raw_laws_set = set(raw_laws['citation'].unique())

# Pull the train citations fresh to ensure this cell runs standalone
train_df = pd.read_csv('/kaggle/input/competitions/llm-agentic-legal-information-retrieval/train.csv')
train_golds = train_df['gold_citations'].dropna().str.split(';').explode().str.strip()

# Isolate just the laws (starts with 'Art.')
train_laws = train_golds[train_golds.str.startswith('Art.', na=False)].unique()

print(f"Checking {len(train_laws)} unique train laws against raw dataset...")

matches = pd.Series(train_laws).isin(raw_laws_set)
matched_count = matches.sum()
missing_count = len(train_laws) - matched_count

print(f"\n--- Train Laws vs Raw laws_de.csv ---")
print(f"Total Unique Train Laws ('Art.'): {len(train_laws)}")
print(f"✅ Found in raw file: {matched_count}")
print(f"❌ Missing from raw file: {missing_count}")

if missing_count > 0:
    missing_examples = pd.Series(train_laws)[~matches].head(10).tolist()
    print(f"\nExamples of laws entirely missing from the base file:")
    for ex in missing_examples:
        print(f"  - {ex}")
    print("\nConclusion: The competition datasets have formatting inconsistencies.")
else:
    print("\nConclusion: All train laws exist in the raw file. Our utility script filtering dropped them during the parquet build.")

Loading raw laws_de.csv (citation column only)...
Checking 2643 unique train laws against raw dataset...

--- Train Laws vs Raw laws_de.csv ---
Total Unique Train Laws ('Art.'): 2643
✅ Found in raw file: 1876
❌ Missing from raw file: 767

Examples of laws entirely missing from the base file:
  - Art. 10a Abs. 1 UVG
  - Art. 975 ZGB
  - Art. 973 ZGB
  - Art. 956a ZGB
  - Art. 976a ZGB
  - Art. 60 OR
  - Art. 264m StGB
  - Art. 50 BV
  - Art. 106 StGB
  - Art. 630 ZGB

Conclusion: The competition datasets have formatting inconsistencies.


🔑 **Finding**: 767 of the missing train laws **never existed in raw
`laws_de.csv` at all**. This is hypothesis B — not a bug in my pipeline,
a property of the dataset.

Examples like `Art. 60 OR`, `Art. 975 ZGB`, `Art. 50 BV` look like normal
citations, but they aren't in the source file the competition gave us.

### 🕵️ But the laws *are* there — just at a different granularity

If `Art. 975 ZGB` isn't in the raw file, what *is* there for Article 975 of
the ZGB? Searching by partial match should reveal whether the law exists
under a different name.

In [11]:

print("Loading raw laws_de.csv...")
raw_df = pd.read_csv('/kaggle/input/competitions/llm-agentic-legal-information-retrieval/laws_de.csv', usecols=['citation', 'title'])

print("\n--- Hunting for '975 ZGB' ---")
# Find any citation containing '975' where the title implies the Zivilgesetzbuch
mask_975 = raw_df['citation'].str.contains('975', regex=False, na=False)
mask_zgb = raw_df['title'].str.contains('Zivilgesetzbuch|ZGB', case=False, na=False)
print(raw_df[mask_975 & mask_zgb].head())

print("\n--- Hunting for '10a' and 'UVG' ---")
# Find '10a' in citation where title implies Unfallversicherungsgesetz
mask_10a = raw_df['citation'].str.contains('10a', regex=False, na=False)
mask_uvg = raw_df['title'].str.contains('Unfallversicherung|UVG', case=False, na=False)
print(raw_df[mask_10a & mask_uvg].head())

print("\n--- Let's look at 5 random matched laws to compare ---")
# For contrast, let's see what a SUCCESSFUL match looks like
train_df = pd.read_csv('/kaggle/input/competitions/llm-agentic-legal-information-retrieval/train.csv')
train_golds = train_df['gold_citations'].dropna().str.split(';').explode().str.strip()
train_laws = train_golds[train_golds.str.startswith('Art.', na=False)].unique()

raw_laws_set = set(raw_df['citation'].unique())
matches = pd.Series(train_laws).isin(raw_laws_set)
print(pd.Series(train_laws)[matches].head(5).tolist())

Loading raw laws_de.csv...

--- Hunting for '975 ZGB' ---
                   citation  \
113669   Art. 975 Abs. 1 OR   
113670   Art. 975 Abs. 2 OR   
172704  Art. 975 Abs. 1 ZGB   
172705  Art. 975 Abs. 2 ZGB   

                                                                                                                                                     title  
113669  Bundesgesetz vom 30. März 1911 betreffend die Ergänzung des Schweizerischen Zivilgesetzbuches (Fünfter Teil: Obligationenrecht) - I.  In der Regel  
113670  Bundesgesetz vom 30. März 1911 betreffend die Ergänzung des Schweizerischen Zivilgesetzbuches (Fünfter Teil: Obligationenrecht) - I.  In der Regel  
172704                                                      Schweizerisches Zivilgesetzbuch vom 10. Dezember 1907 - II.  Bei ungerechtfertigtem Eintrag719  
172705                                                      Schweizerisches Zivilgesetzbuch vom 10. Dezember 1907 - II.  Bei ungerechtfertigtem Eintrag719  



🔑 **Finding**: `Art. 975 ZGB` isn't in the corpus, but `Art. 975 Abs. 1 ZGB`
and `Art. 975 Abs. 2 ZGB` are. The corpus stores laws at paragraph-level;
train sometimes cites at article-level.

Confirmed by the host (Ari) on the forum:
> *"LeXam doesn't have a consistent normalization. But we always put the
> paragraphs like 'Art. 60 Abs. 2 OR' when they exist."*

**Decision**: build a **paragraph-aware matcher** — `'Art. 975 ZGB'` should
count as a hit for any of its paragraph-level entries. Diagnostic below
quantifies the win.

In [12]:
# Diagnostic: classify the 767 missing train laws into recoverable vs unreachable buckets
import pandas as pd
import re
from collections import defaultdict

print("Loading raw laws_de.csv (citation column)...")
raw_df = pd.read_csv(
    '/kaggle/input/competitions/llm-agentic-legal-information-retrieval/laws_de.csv',
    usecols=['citation']
)
raw_citations = raw_df['citation'].dropna().unique()
raw_set = set(raw_citations)
print(f"Raw corpus has {len(raw_set):,} unique citations")

# Re-fetch train laws fresh
train_df = pd.read_csv('/kaggle/input/competitions/llm-agentic-legal-information-retrieval/train.csv')
train_golds = train_df['gold_citations'].dropna().str.split(';').explode().str.strip()
train_laws = train_golds[train_golds.str.startswith('Art.', na=False)].unique()

# Identify the missing ones
missing = [c for c in train_laws if c not in raw_set]
print(f"Missing from raw corpus: {len(missing)} unique citations\n")

# Build a lookup: for each (article_number, law_code) tuple, what paragraph-level
# entries exist in the raw corpus?
# Pattern: "Art. <num>[a-z]? [Abs. <num>] <CODE>"
paragraph_index = defaultdict(list)
article_pattern = re.compile(r'^Art\.\s+(\S+)\s+(?:Abs\.\s+\S+\s+)?(\S+)$')

for c in raw_citations:
    m = article_pattern.match(c)
    if m:
        art_num, law_code = m.group(1), m.group(2)
        paragraph_index[(art_num, law_code)].append(c)

# Now classify each missing train law
def classify(citation):
    """Return ('recoverable', list_of_corpus_matches) or ('unreachable', [])."""
    m = article_pattern.match(citation)
    if not m:
        return ('malformed', [])
    art_num, law_code = m.group(1), m.group(2)
    matches = paragraph_index.get((art_num, law_code), [])
    if matches:
        return ('recoverable', matches)
    return ('unreachable', [])

bucket_recoverable = []   # article-level gold, paragraph-level corpus entries exist
bucket_unreachable = []   # not in corpus at any granularity
bucket_malformed = []     # didn't match expected pattern

for c in missing:
    category, matches = classify(c)
    if category == 'recoverable':
        bucket_recoverable.append((c, matches))
    elif category == 'unreachable':
        bucket_unreachable.append(c)
    else:
        bucket_malformed.append(c)

# Report
total = len(missing)
print(f"=== Breakdown of {total} missing train laws ===\n")

print(f"📗 RECOVERABLE (article-level gold, paragraph-level in corpus): {len(bucket_recoverable)} ({100*len(bucket_recoverable)/total:.1f}%)")
print(f"   These will match once we use a paragraph-aware matcher.")
print(f"   Examples:")
for gold, corpus_matches in bucket_recoverable[:5]:
    print(f"     {gold!r}  →  corpus has: {corpus_matches}")

print(f"\n📕 UNREACHABLE (not in corpus at any granularity): {len(bucket_unreachable)} ({100*len(bucket_unreachable)/total:.1f}%)")
print(f"   These cap our F1 ceiling — no retrieval system can find them.")
print(f"   Examples:")
for c in bucket_unreachable[:10]:
    print(f"     {c!r}")

if bucket_malformed:
    print(f"\n📙 MALFORMED (didn't match expected pattern): {len(bucket_malformed)}")
    print(f"   Examples:")
    for c in bucket_malformed[:5]:
        print(f"     {c!r}")

# Final ceiling calculation
total_train_laws = len(train_laws)
matched_directly = total_train_laws - total  # 1876
recoverable = len(bucket_recoverable)
truly_unreachable = len(bucket_unreachable) + len(bucket_malformed)

print(f"\n=== F1 Ceiling Analysis ===")
print(f"Total unique train laws:        {total_train_laws}")
print(f"  Direct match in corpus:       {matched_directly} ({100*matched_directly/total_train_laws:.1f}%)")
print(f"  Recoverable via paragraph match: {recoverable} ({100*recoverable/total_train_laws:.1f}%)")
print(f"  Unreachable:                  {truly_unreachable} ({100*truly_unreachable/total_train_laws:.1f}%)")
print(f"\n  Theoretical max coverage with smart matcher: {100*(matched_directly + recoverable)/total_train_laws:.1f}%")

Loading raw laws_de.csv (citation column)...
Raw corpus has 175,933 unique citations
Missing from raw corpus: 767 unique citations

=== Breakdown of 767 missing train laws ===

📗 RECOVERABLE (article-level gold, paragraph-level in corpus): 713 (93.0%)
   These will match once we use a paragraph-aware matcher.
   Examples:
     'Art. 975 ZGB'  →  corpus has: ['Art. 975 Abs. 1 ZGB', 'Art. 975 Abs. 2 ZGB']
     'Art. 973 ZGB'  →  corpus has: ['Art. 973 Abs. 1 ZGB', 'Art. 973 Abs. 2 ZGB']
     'Art. 956a ZGB'  →  corpus has: ['Art. 956a Abs. 1 ZGB', 'Art. 956a Abs. 2 ZGB', 'Art. 956a Abs. 3 ZGB']
     'Art. 976a ZGB'  →  corpus has: ['Art. 976a Abs. 1 ZGB', 'Art. 976a Abs. 2 ZGB']
     'Art. 60 OR'  →  corpus has: ['Art. 60 Abs. 1 OR', 'Art. 60 Abs. 1bis OR', 'Art. 60 Abs. 2 OR', 'Art. 60 Abs. 3 OR']

📕 UNREACHABLE (not in corpus at any granularity): 53 (6.9%)
   These cap our F1 ceiling — no retrieval system can find them.
   Examples:
     'Art. 10a Abs. 1 UVG'
     'Art. 257 ZPO'
     '

🔑 **Finding**: 93% of missing train citations are recoverable, only 2% are
truly unreachable (mostly international treaties like `LugÜ`).

**F1 ceiling on train laws: 71% → 98%** with the paragraph-aware matcher.
Implement it in Phase 6 (threshold tuning).